In [1]:
# =========================================
# 1. IMPORT LIBRARIES
# =========================================
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt

# =========================================
# 2. LOAD DATA
# =========================================
data = pd.read_csv('data.csv')

print("Initial Missing Values:\n", data.isnull().sum())

# =========================================
# 3. HANDLE MISSING VALUES
# =========================================

# Numerical columns
data['Income'].fillna(data['Income'].median(), inplace=True)
data['Credit_Score'].fillna(data['Credit_Score'].mean(), inplace=True)
data['Loan_Balance'].fillna(data['Loan_Balance'].median(), inplace=True)
data['Credit_Utilization'].fillna(data['Credit_Utilization'].mean(), inplace=True)
data['Debt_to_Income_Ratio'].fillna(data['Debt_to_Income_Ratio'].mean(), inplace=True)

# Payment history (categorical)
months = ['Month_1','Month_2','Month_3','Month_4','Month_5','Month_6']
for col in months:
    data[col].fillna(data[col].mode()[0], inplace=True)

# =========================================
# 4. HANDLE ANOMALIES
# =========================================

# Remove negative income
data = data[data['Income'] > 0]

# Clip Credit Utilization (0–100)
data['Credit_Utilization'] = data['Credit_Utilization'].clip(0, 100)

# Handle Loan Balance outliers (IQR)
Q1 = data['Loan_Balance'].quantile(0.25)
Q3 = data['Loan_Balance'].quantile(0.75)
IQR = Q3 - Q1

lower = Q1 - 1.5 * IQR
upper = Q3 + 1.5 * IQR

data['Loan_Balance'] = data['Loan_Balance'].clip(lower, upper)

print("\nMissing Values After Cleaning:\n", data.isnull().sum())

# =========================================
# 5. ENCODE CATEGORICAL VARIABLES
# =========================================
data = pd.get_dummies(data, columns=[
    'Employment_Status',
    'Credit_Card_Type',
    'Location'
], drop_first=True)

# Drop ID column
data.drop('Customer_ID', axis=1, inplace=True)

# =========================================
# 6. EDA (VISUALIZATION)
# =========================================

# Correlation Heatmap
plt.figure(figsize=(10,8))
sns.heatmap(data.corr(), cmap='coolwarm')
plt.title("Correlation Heatmap")
plt.show()

# Boxplots
sns.boxplot(x='Delinquent_Account', y='Credit_Utilization', data=data)
plt.title("Credit Utilization vs Delinquency")
plt.show()

sns.boxplot(x='Delinquent_Account', y='Missed_Payments', data=data)
plt.title("Missed Payments vs Delinquency")
plt.show()

sns.boxplot(x='Delinquent_Account', y='Income', data=data)
plt.title("Income vs Delinquency")
plt.show()

# =========================================
# 7. PREPARE DATA
# =========================================
X = data.drop('Delinquent_Account', axis=1)
y = data['Delinquent_Account']

# =========================================
# 8. TRAIN-TEST SPLIT
# =========================================
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

# =========================================
# 9. LOGISTIC REGRESSION
# =========================================
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report

model = LogisticRegression(max_iter=1000)
model.fit(X_train, y_train)

y_pred = model.predict(X_test)

print("\nLogistic Regression:")
print("Accuracy:", accuracy_score(y_test, y_pred))
print(classification_report(y_test, y_pred))

# =========================================
# 10. RANDOM FOREST
# =========================================
from sklearn.ensemble import RandomForestClassifier

rf = RandomForestClassifier()
rf.fit(X_train, y_train)

y_pred_rf = rf.predict(X_test)

print("\nRandom Forest:")
print("Accuracy:", accuracy_score(y_test, y_pred_rf))
print(classification_report(y_test, y_pred_rf))

# =========================================
# 11. FEATURE IMPORTANCE
# =========================================
importances = rf.feature_importances_
features = X.columns

feat_importance = pd.Series(importances, index=features).sort_values(ascending=False)

plt.figure(figsize=(10,6))
feat_importance.head(10).plot(kind='barh')
plt.title("Top 10 Important Features")
plt.show()

Initial Missing Values:
 Customer_ID              0
Age                      0
Income                  39
Credit_Score             2
Credit_Utilization       0
Missed_Payments          0
Delinquent_Account       0
Loan_Balance            29
Debt_to_Income_Ratio     0
Employment_Status        0
Account_Tenure           0
Credit_Card_Type         0
Location                 0
Month_1                  0
Month_2                  0
Month_3                  0
Month_4                  0
Month_5                  0
Month_6                  0
dtype: int64

Missing Values After Cleaning:
 Customer_ID              0
Age                      0
Income                   0
Credit_Score             1
Credit_Utilization       0
Missed_Payments          0
Delinquent_Account       0
Loan_Balance            25
Debt_to_Income_Ratio     0
Employment_Status        0
Account_Tenure           0
Credit_Card_Type         0
Location                 0
Month_1                  0
Month_2                  0
Month_3    

C:\Users\vnd31\AppData\Local\Temp\ipykernel_27932\99736999.py:21: ChainedAssignmentError: A value is being set on a copy of a DataFrame or Series through chained assignment using an inplace method.
Such inplace method never works to update the original DataFrame or Series, because the intermediate object on which we are setting values always behaves as a copy (due to Copy-on-Write).

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' instead, to perform the operation inplace on the original object, or try to avoid an inplace operation using 'df[col] = df[col].method(value)'.

See the documentation for a more detailed explanation: https://pandas.pydata.org/pandas-docs/stable/user_guide/copy_on_write.html
  data['Income'].fillna(data['Income'].median(), inplace=True)
C:\Users\vnd31\AppData\Local\Temp\ipykernel_27932\99736999.py:22: ChainedAssignmentError: A value is being set on a copy of a DataFrame or Series through chained 

ValueError: could not convert string to float: 'Late'

<Figure size 1000x800 with 0 Axes>